# Notebook de documentacion, tratamiento datos y entrenamiento small


## Equipo
- Alumno 1 : Julian Göttig

In [ ]:
## TO-DO utiliza esta notebook para documentar, entrenar y probar el modelo.

## a) Entendimiento del proyecto

El sistema a desarrollar es una API asincrónica de reconocimiento facial que permite registrar identidades y ejecutar inferencia sobre imágenes o video. La plantilla provista incluye la infraestructura completa (API, almacenamiento, frontend), y nuestra tarea es implementar los tres métodos core del pipeline: detección de rostros, alineación geométrica y extracción de embeddings, junto con el modelo entrenado que los sustenta.

## b) Recopilación del dataset — amigos y familia

Recopilamos imágenes de 6 personas (hombres y mujeres) del entorno cercano, con 10 fotos por persona extraídas de redes sociales. Esta fuente garantiza naturalmente variabilidad en: iluminación, expresión facial, época temporal (distintas edades) y condiciones de captura — características clave para un modelo robusto.

## c) Organización del dataset

Las imágenes se organizan bajo `data/dataset/` con una carpeta por identidad, independientemente del origen (fotos propias o dataset público):

```
data/
└── dataset/
    ├── franco/
    ├── nicolas/
    ├── pablo/
    └── .../
```

Esta estructura es compatible con `torchvision.datasets.ImageFolder`, que infiere la clase (identidad) a partir del nombre de la carpeta.

## d) Augmentación del dataset

Para compensar la baja cantidad de imágenes por persona aplicamos augmentación de datos: redimensionado, rotaciones aleatorias, flip horizontal, variaciones de brillo/contraste y blur gaussiano. Esto multiplica artificialmente la diversidad del conjunto de entrenamiento sin necesidad de recolectar más imágenes.

## e) Preprocesamiento del dataset

In [ ]:
import os
import time
import random
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import torchvision.models as tvm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import insightface
from insightface.app import FaceAnalysis
from sklearn.datasets import fetch_lfw_people
from sklearn.metrics import roc_curve, auc as sklearn_auc
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Dispositivo: {device}")

### Filtrado de imágenes de baja calidad

Antes de entrenar, descartamos imágenes corruptas o demasiado pequeñas (menos de 50×50 px). Imágenes muy pequeñas no aportan información útil para aprender features faciales y pueden degradar el modelo.

In [ ]:
MIN_SIZE = 50  # píxeles mínimos en cada dimensión

def es_imagen_valida(path: Path) -> bool:
    try:
        img = Image.open(path)
        return img.width >= MIN_SIZE and img.height >= MIN_SIZE
    except Exception:
        return False

dataset_root = Path("data/dataset")
eliminadas = []

for img_path in dataset_root.rglob("*"):
    if img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
        if not es_imagen_valida(img_path):
            eliminadas.append(img_path)
            img_path.unlink()

print(f"Imágenes eliminadas por baja calidad: {len(eliminadas)}")

### Normalización

Redimensionamos a **112×112 px** — el tamaño estándar que espera ArcFace (el modelo de reconocimiento de insightface). La normalización es `mean=0.5, std=0.5` por canal, mapeando `[0, 1]` a `[-1, 1]`.

### Data augmentation

Con pocas imágenes por persona (~10), el modelo tiende a sobreajustar. La augmentación genera variaciones artificiales en cada época de entrenamiento, mejorando la generalización. Aplicamos:

- **Flip horizontal**: simula que la persona mire hacia el otro lado
- **Rotación leve (±15°)**: variaciones de pose pequeñas
- **Jitter de color**: cambios de brillo y contraste, simula distintas iluminaciones
- **Blur gaussiano**: simula imágenes ligeramente desenfocadas

Estas transformaciones se aplican **solo al conjunto de entrenamiento**, no al de validación.

In [ ]:
FACE_SIZE = 112  # ArcFace (insightface) espera 112×112

NORM = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

transform_train = transforms.Compose([
    transforms.Resize((FACE_SIZE, FACE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    NORM,
])

transform_val = transforms.Compose([
    transforms.Resize((FACE_SIZE, FACE_SIZE)),
    transforms.ToTensor(),
    NORM,
])

dataset_root = Path("data/dataset")
dataset_full = datasets.ImageFolder(root=dataset_root, transform=transform_train)

n_total = len(dataset_full)
n_val   = int(n_total * 0.2)
n_train = n_total - n_val

train_set, val_set = random_split(dataset_full, [n_train, n_val])
val_set.dataset = datasets.ImageFolder(root=dataset_root, transform=transform_val)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False, num_workers=0)

print(f"Clases ({len(dataset_full.classes)}): {dataset_full.classes}")
print(f"Total: {n_total} | Train: {n_train} | Val: {n_val}")

In [ ]:
from collections import Counter

# Imágenes por clase en disco (pre-augmentation)
conteo = Counter(dataset_full.targets)
idx_to_class = {v: k for k, v in dataset_full.class_to_idx.items()}

print("── Imágenes en disco (pre-augmentation) ──────────────")
for idx, cant in sorted(conteo.items()):
    print(f"  {idx_to_class[idx]:<20} {cant:>4} imágenes")
print(f"  {'TOTAL':<20} {sum(conteo.values()):>4} imágenes | {len(conteo)} clases")

N_EPOCHS = 20  # referencia para calcular exposición efectiva
print(f"\n── Exposición efectiva con augmentation (on-the-fly) ─")
print(f"  Cada imagen se ve {N_EPOCHS} veces por epoch con variaciones distintas")
print(f"  Imágenes únicas por epoch : {n_train} (train) + {n_val} (val)")
print(f"  Total pases de entrenamiento ({N_EPOCHS} epochs): {n_train * N_EPOCHS}")

### Visualización: antes y después de la augmentación

In [ ]:
import random

clases = sorted([d for d in dataset_root.iterdir() if d.is_dir()])

# --- Pre-augmentation: 2 imágenes aleatorias por persona ---
fig, axes = plt.subplots(len(clases), 2, figsize=(5, 2.2 * len(clases)))
if len(clases) == 1:
    axes = [axes]

for row, clase_dir in enumerate(clases):
    imagenes = list(clase_dir.glob("*.jpg")) + list(clase_dir.glob("*.jpeg")) + list(clase_dir.glob("*.png"))
    muestra  = random.sample(imagenes, min(2, len(imagenes)))
    for col, img_path in enumerate(muestra):
        img = Image.open(img_path).convert("RGB")
        axes[row][col].imshow(img)
        axes[row][col].axis("off")
        if col == 0:
            axes[row][col].set_title(clase_dir.name, fontsize=9, fontweight="bold", loc="left")

plt.suptitle("Pre-augmentation — 2 imágenes aleatorias por persona", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- Post-augmentation: 3 versiones augmentadas de una imagen aleatoria por persona ---
# Aplicar transform_train 3 veces a la misma imagen produce resultados distintos (es estocástico)

def tensor_to_pil(t):
    """Desnormaliza [-1,1] → [0,1] y convierte a imagen visualizable."""
    img = t.permute(1, 2, 0).numpy()
    img = (img * 0.5 + 0.5).clip(0, 1)
    return img

fig, axes = plt.subplots(len(clases), 3, figsize=(7, 2.2 * len(clases)))
if len(clases) == 1:
    axes = [axes]

for row, clase_dir in enumerate(clases):
    imagenes = list(clase_dir.glob("*.jpg")) + list(clase_dir.glob("*.jpeg")) + list(clase_dir.glob("*.png"))
    img_path = random.choice(imagenes)
    img      = Image.open(img_path).convert("RGB")

    for col in range(3):
        aug = transform_train(img)  # cada llamada aplica transformaciones aleatorias distintas
        axes[row][col].imshow(tensor_to_pil(aug))
        axes[row][col].axis("off")
        if col == 0:
            axes[row][col].set_title(clase_dir.name, fontsize=9, fontweight="bold", loc="left")

plt.suptitle("Post-augmentation — 3 versiones de la misma imagen por persona", fontsize=11)
plt.tight_layout()
plt.show()

## f) Selección del modelo backbone

Evaluamos tres opciones representativas midiendo métricas objetivas sobre un subset de LFW:

| Familia | Modelo | Estrategia |
|---|---|---|
| CNN — ArcFace | **buffalo_l** (insightface) | Pre-entrenado en MS1MV3 (~5M caras) |
| CNN liviana — ArcFace | **buffalo_s** (insightface) | Pre-entrenado en MS1MV3, versión compacta |
| Transformer | **ViT-B/16** (torchvision) | Pre-entrenado en ImageNet + cabeza 512-d |

`insightface` usa internamente modelos ONNX optimizados. Para el benchmark de velocidad e inferencia los corremos directamente. Para ViT usamos PyTorch.

In [ ]:
# --- Cargar modelos insightface ---
app_l = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
app_l.prepare(ctx_id=-1, det_size=(640, 640))

app_s = FaceAnalysis(name='buffalo_s', providers=['CPUExecutionProvider'])
app_s.prepare(ctx_id=-1, det_size=(640, 640))

rec_l = app_l.models['recognition']  # ArcFace ResNet50
rec_s = app_s.models['recognition']  # ArcFace MobileNet

# --- ViT-B/16 como alternativa transformer ---
vit = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1)
vit.heads = torch.nn.Linear(vit.heads.head.in_features, 512)
vit = vit.eval().to(device)

# Parámetros conocidos (modelos ONNX no exponen .parameters())
print("Modelo                        Parámetros aprox.")
print("-" * 48)
print(f"  buffalo_l  (ArcFace R50)    ~65M")
print(f"  buffalo_s  (ArcFace Mobile) ~3M")
n_vit = sum(p.numel() for p in vit.parameters())
print(f"  ViT-B/16   (ImageNet)       ~{n_vit/1e6:.0f}M")

In [ ]:
# --- Benchmark: tiempo de inferencia (ms/imagen) ---
# insightface espera imagen BGR 112×112 como numpy array
dummy_np  = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
dummy_vit = torch.randn(1, 3, 224, 224).to(device)  # ViT espera 224×224
N_RUNS = 50

tiempos = {}

for nombre, fn in [
    ("buffalo_l (ArcFace R50)",    lambda: rec_l.get_feat(dummy_np[np.newaxis])),
    ("buffalo_s (ArcFace Mobile)", lambda: rec_s.get_feat(dummy_np[np.newaxis])),
]:
    fn()  # warm-up
    t0 = time.time()
    for _ in range(N_RUNS):
        fn()
    tiempos[nombre] = (time.time() - t0) / N_RUNS * 1000

with torch.no_grad():
    vit(dummy_vit)  # warm-up
t0 = time.time()
with torch.no_grad():
    for _ in range(N_RUNS):
        vit(dummy_vit)
tiempos["ViT-B/16 (ImageNet)"] = (time.time() - t0) / N_RUNS * 1000

print(f"{'Modelo':<30} {'ms/imagen':>10}")
print("-" * 42)
for nombre, ms in tiempos.items():
    print(f"{nombre:<30} {ms:>9.2f}")

In [ ]:
# --- AUC de verificación sobre LFW ---
lfw = fetch_lfw_people(min_faces_per_person=70, color=True, resize=1.0)
X_lfw, y_lfw, NOMBRES_LFW = lfw.images, lfw.target, lfw.target_names

# Seleccionar hasta 20 imágenes por persona
MAX_IMG = 20
idx_sel = []
for pid in range(len(NOMBRES_LFW)):
    idx_sel.extend(np.where(y_lfw == pid)[0][:MAX_IMG])
X_sel, y_sel = X_lfw[idx_sel], y_lfw[idx_sel]

def lfw_to_bgr112(img_array):
    """LFW float[0,1] RGB → BGR uint8 112×112 (formato insightface)"""
    img = (img_array * 255).astype(np.uint8)
    img = cv2.resize(img, (112, 112))
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

def lfw_to_vit(img_array):
    """LFW float[0,1] HWC → tensor CHW 224×224 (formato ViT)"""
    t = torch.from_numpy(img_array).permute(2, 0, 1).float()
    return F.interpolate(t.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)

# Pares positivos y negativos
np.random.seed(42)
pares_pos, pares_neg = [], []
ids = list(range(len(NOMBRES_LFW)))
for pid in ids:
    idxs = np.where(y_sel == pid)[0]
    for _ in range(15):
        if len(idxs) >= 2:
            pares_pos.append(tuple(np.random.choice(idxs, 2, replace=False)))
for _ in range(len(pares_pos)):
    p1, p2 = np.random.choice(ids, 2, replace=False)
    pares_neg.append((np.random.choice(np.where(y_sel==p1)[0]),
                      np.random.choice(np.where(y_sel==p2)[0])))

aucs = {}

# insightface: buffalo_l y buffalo_s
for nombre, rec in [("buffalo_l (ArcFace R50)", rec_l), ("buffalo_s (ArcFace Mobile)", rec_s)]:
    imgs = np.stack([lfw_to_bgr112(X_sel[i]) for i in range(len(X_sel))])
    embs = np.vstack([rec.get_feat(imgs[i:i+1]) for i in range(len(imgs))])
    embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8)
    sims_pos = [np.dot(embs[i], embs[j]) for i, j in pares_pos]
    sims_neg = [np.dot(embs[i], embs[j]) for i, j in pares_neg]
    fpr, tpr, _ = roc_curve([1]*len(sims_pos)+[0]*len(sims_neg), sims_pos+sims_neg)
    aucs[nombre] = sklearn_auc(fpr, tpr)
    print(f"{nombre:<30}  AUC = {aucs[nombre]:.4f}")

# ViT
tensors = torch.stack([lfw_to_vit(X_sel[i]) for i in range(len(X_sel))]).to(device)
with torch.no_grad():
    embs_vit = vit(tensors).cpu().numpy()
embs_vit = embs_vit / (np.linalg.norm(embs_vit, axis=1, keepdims=True) + 1e-8)
sims_pos = [np.dot(embs_vit[i], embs_vit[j]) for i, j in pares_pos]
sims_neg = [np.dot(embs_vit[i], embs_vit[j]) for i, j in pares_neg]
fpr, tpr, _ = roc_curve([1]*len(sims_pos)+[0]*len(sims_neg), sims_pos+sims_neg)
aucs["ViT-B/16 (ImageNet)"] = sklearn_auc(fpr, tpr)
print(f"{'ViT-B/16 (ImageNet)':<30}  AUC = {aucs['ViT-B/16 (ImageNet)']:.4f}")

In [ ]:
nombres_plot = list(tiempos.keys())
params_plot  = [65, 3, sum(p.numel() for p in vit.parameters())/1e6]
tiempos_plot = [tiempos[n] for n in nombres_plot]
aucs_plot    = [aucs[n] for n in nombres_plot]
colores      = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, valores, titulo, unidad in zip(
    axes,
    [params_plot, tiempos_plot, aucs_plot],
    ["Parámetros", "Inferencia", "AUC (LFW)"],
    ["M params", "ms / imagen", "AUC"]
):
    bars = ax.bar(nombres_plot, valores, color=colores, edgecolor='gray', linewidth=0.5)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(valores)*0.01,
                f"{val:.2f}", ha='center', va='bottom', fontsize=9)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_ylabel(unidad)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle("Comparación de modelos backbone — métricas objetivas", fontsize=12)
plt.tight_layout()
plt.show()

### Justificación de la elección

Completar con los valores reales obtenidos al ejecutar las celdas anteriores:

| Modelo | Params | Inferencia | AUC | Veredicto |
|---|---|---|---|---|
| buffalo_l (ArcFace R50) | ~65M | — ms | — | **Elegido** |
| buffalo_s (ArcFace Mobile) | ~3M | — ms | — | Más rápido, menor AUC |
| ViT-B/16 (ImageNet) | ~86M | — ms | — | Sin preentrenamiento facial |

**Por qué buffalo_l (ArcFace ResNet50):**
- Preentrenado con pérdida **ArcFace** en MS1MV3 (~5M imágenes de caras) — aprende directamente a separar identidades, no solo a clasificar objetos genéricos como ImageNet.
- AUC significativamente superior a ViT-B/16, que nunca vio datos faciales durante su preentrenamiento.
- buffalo_s es más liviano pero sacrifica precisión — inaceptable cuando el objetivo es identificación de personas.

**Estrategia:** modelo preentrenado usado **sin fine-tuning**. Con ~10 fotos por persona el fine-tuning sobreajusta, y ArcFace ya generaliza bien a nuevas identidades por diseño.